<p align="center">
  <img src="https://raw.githubusercontent.com/paulmunozpauta/Hidrology_Course/main/Static/Imgs/UdeC_color_horizontal.jpg" width="500">
</p>
<p align="center"><b style="font-size:28px;">Facultad de Ingeniería Agrícola</b></p>
<p align="center"><b style="font-size:28px;">Curso de Hidrología</b></p>
<hr>
<p align="center"><b style="font-size:28px;">Contacto</b></p>
<p align="center">
  paulmunoz@udec.cl<br>
  https://paulmunoz.com
</p>

# Probabilidad y estadística

## Objetivo

Aplicar fundamentos de probabilidad y estadística en una serie temporal real de Chile



## Clonar el repositorio desde GitHub

In [ ]:
# Clonar el repositorio desde GitHub
!git clone -- https://github.com/paulmunozpauta/Hidrology_Course

Después de ejecutar la celda anterior, se crea una carpeta en nuestro entorno llamada:

**Hidrology_Course**

Esta carpeta contiene todos los archivos del curso.

Ahora debemos entrar a esa carpeta para trabajar con los datos.

# Entrar a la carpeta del repositorio





In [ ]:
# Entrar a la carpeta del repositorio
%cd Hidrology_Course
!ls

##Instalación e importación de librerías


Librerías principales:
pandas
numpy
matplotlib

In [ ]:
# Instalar librerías necesarias
!pip -q install pandas

In [ ]:
#Importar librerías
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Lectura  de datos hidrometerológicos

Los datos a usar están disponibles de forma libre en: https://explorador.cr2.cl/

Vamos a usar los datos de la estación Puerto Montt de la Dirección General de Aguas de Chile (10425001)

In [ ]:
# ruta relativa del archivo en el repositorio
archivo = "Data/Precipitación/PuertoMontt_DGA_10425001.xlsx"
# leer el archivo Excel original de la DGA
df = pd.read_excel(archivo)
df

# Preprocesamiento (preparación) de datos para los análisis

Los datos diarios de esta estación están separados en columnas por "agno", "mes", "día" y "valor".

Vamos a generar una serie (dataframe) ordenada cronológicamente a través de la fecha disponible

In [ ]:
# limpiar nombres de columnas
df.columns = df.columns.str.strip().str.lower()

# renombrar por seguridad si vienen con espacios o acentos raros
df = df.rename(columns={
    "agno": "anio",
    "año": "anio",
    "mes": "mes",
    "dia": "dia",
    "valor": "precipitacion_diaria_mm"
})

# convertir columnas a numéricas
for col in ["anio", "mes", "dia", "precipitacion_diaria_mm"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# eliminar filas con datos faltantes en columnas clave
df = df.dropna(subset=["anio", "mes", "dia", "precipitacion_diaria_mm"])

# convertir año, mes y día a enteros
df["anio"] = df["anio"].astype(int)
df["mes"] = df["mes"].astype(int)
df["dia"] = df["dia"].astype(int)

# crear columna fecha
df["fecha"] = pd.to_datetime(
    dict(year=df["anio"], month=df["mes"], day=df["dia"]),
    errors="coerce"
)

# eliminar fechas inválidas
df = df.dropna(subset=["fecha"])

# dejar solo la serie temporal final
ts = df[["fecha", "precipitacion_diaria_mm"]].copy()

# ordenar por fecha
ts = ts.sort_values("fecha").reset_index(drop=True)

# opcional: usar fecha como índice
ts = ts.set_index("fecha")

# mostrar la serie
ts


## Figura de de precipitación diaria



Graficar esta serie preprocesada mediante barras, ya que la precipitación es una acumulación en cada día

In [ ]:
plt.figure(figsize=(12,5))

ts.plot()
plt.xlabel("Fecha")
plt.ylabel("Precipitación (mm)")
plt.grid(axis="y")

plt.show()


## Probabilidad de ocurrencia y tiempo de retorno

Vamos a calcular la probabilidad de ocurrencia y el periodo de retorno de precipitaciones extremas en esta serie.

Para ello, partimos de una serie de duración completa (datos diarios de la DGA) y la transformamos en una serie de valores extremos anuales.

Esto significa que, para cada año, seleccionamos el valor máximo de precipitación.

Pasos del análisis:
- Generar la serie de valores extremos anuales
- Aplicar la probabilidad empírica de Weibull para estimar la probabilidad de cada valor de precipitación
- A partir de la probabilidad, estimar el periodo de retorno de las distintas magnitudes de precipitación

### Precipitación **máxima diaria anual**

Vamos a generar una serie en donde tengamos un valor de precipitación por año, en cada año este valor de precipitación corresponde a la precipitación máxima del año.

Además vamos a depurar la serie para que sólo se genere un datos por año para cuando tengo más del 90 % de los datos, es decir (menos o igual al 10 % de vacíos en el número de días del año).

Si en un año no tenemos el 90% de datos, entonces la serie escribirá "Not a Number" (NaN), que significa que python le reconoce como un dato no numérico.

In [ ]:

# conteos
dias_validos = ts["precipitacion_diaria_mm"].resample("YE").count()
dias_totales = ts["precipitacion_diaria_mm"].resample("YE").size()

# fracción faltante
fraccion_faltante = 1 - (dias_validos / dias_totales)

# máximo anual
max_anual = ts[col].resample("YE").max()

# filtro calidad
max_anual = max_anual.where(fraccion_faltante <= 0.10)

# DataFrame
max_anual = max_anual.to_frame(name="precipitacion_max_diaria_anual_mm")

print(max_anual)

Ahora graficamos la precipitación **máxima diaria anual**.


In [ ]:
plt.figure(figsize=(10,5))

plt.bar(
    max_anual.index.year,
    max_anual["precipitacion_max_diaria_anual_mm"]
)

plt.title("Máximo diario anual")
plt.xlabel("Año")
plt.ylabel("mm")

plt.grid(axis="y")
plt.show()

## Probabilidad **empírica** de Weibull y tiempo de retorno

La fórmula de Weibull permite estimar probabilidades directamente desde los datos, sin asumir una distribución teórica.
	​
Para estimar la probabilidad de ocurrencia de eventos extremos, se utiliza la **fórmula empírica de Weibull**, que asigna una probabilidad a cada valor ordenado de la serie.

$$
P = \frac{m}{n + 1}
$$

### ¿Qué significa cada término?

- $P$: probabilidad de excedencia  
- $m$: posición del dato en la serie ordenada (ranking)  
- $n$: número total de datos  



### ¿Cómo se aplica?

1. Ordenar los datos de precipitación extrema de **mayor a menor**  
2. Asignar a cada valor un ranking $m = 1, 2, 3, ..., n$  
3. Calcular la probabilidad usando la fórmula  



### Interpretación

- Valores con **mayor magnitud** → menor probabilidad  
- Valores más frecuentes → mayor probabilidad  

### Relación con el periodo de retorno

A partir de la probabilidad, el periodo de retorno se calcula como:

$$
T = \frac{1}{P}
$$

---




In [ ]:
serie = max_anual["precipitacion_max_diaria_anual_mm"].dropna()

# ordenar
serie = serie.sort_values(ascending=False).reset_index(drop=True)

n = len(serie)
m = np.arange(1, n+1)

T = (n + 1) / m
P = 1 / T

df_weibull = pd.DataFrame({
    "precipitacion_mm": serie,
    "rank": m,
    "T": T,
    "P_excedencia": P
})

print(df_weibull)

## Curva de frecuencia de precipitación (Weibull)

Esta curva es fundamental para el diseño hidráulico, ya que permite relacionar la **intensidad del evento** con su **frecuencia de ocurrencia**.

Sin embargo, hay que recordart que calculamos una probabilidad empírica, no hemos ajustado una función dprobabilidad por lo que no podemos extrapolar a valores fuera del rango histórico.

 La curva de frecuencia de precipitación representa la relación entre la **precipitación extrema** y su **periodo de retorno**.

- Cada punto corresponde a un evento extremo anual  
- En el eje X se muestra el **periodo de retorno (T)** en escala logarítmica  
- En el eje Y se muestra la **magnitud de precipitación (mm)**  


### ¿Por qué usamos escala logarítmica?

El periodo de retorno puede variar mucho (por ejemplo, de 2 a 100 años), por lo que la escala logarítmica permite:

- Visualizar mejor todo el rango de valores  
- Interpretar más fácilmente eventos frecuentes y extremos en la misma gráfica  


### Interpretación

- Eventos con **mayor precipitación** tienen **mayor periodo de retorno**  
- Eventos más extremos son **menos frecuentes**  
- La curva permite estimar:



In [ ]:
plt.figure(figsize=(7,5))

plt.scatter(df_weibull["T"], df_weibull["precipitacion_mm"])

plt.xscale("log")

plt.xlabel("Periodo de retorno (años)")
plt.ylabel("Precipitación (mm)")
plt.title("Curva de frecuencia - Weibull")

plt.grid(True)
plt.show()

A partir de la curva de frecuencia anterior, surgen preguntas importantes:

- ¿Qué precipitación corresponde a un periodo de retorno dado?  
- ¿Cada cuántos años ocurre una precipitación de cierta magnitud?  


Para responder estas preguntas de forma más completa, es útil analizar la **distribución de los datos**.

Por ello, a continuación se grafica el **histograma de la precipitación máxima diaria anual**, lo que permite:

- Visualizar cómo se distribuyen los eventos extremos  
- Identificar valores más frecuentes y valores extremos  
- Comprender mejor la forma de la distribución (asimetría, dispersión)  



Histograma de la precipitación máxima diaria anual

In [ ]:
plt.figure(figsize=(7,5))

plt.hist(df_weibull["precipitacion_mm"], bins=10, edgecolor='black')

plt.xlabel("Precipitación máxima diaria anual (mm)")
plt.ylabel("Frecuencia")
plt.title("Distribución de precipitaciones extremas")

plt.grid(True, linestyle="--", alpha=0.6)

plt.show()

## Estimación de precipitaciones para periodos de retorno específicos

Para determinar qué precipitación corresponde a un periodo de retorno dado (por ejemplo, 2, 5 y 10 años), se utiliza interpolación sobre la curva de frecuencia obtenida previamente.

Dado que los valores de periodo de retorno calculados empíricamente no coinciden exactamente con estos valores, la interpolación permite estimar la precipitación asociada de manera continua.


### Interpretación

- T = 2 años → evento frecuente  
- T = 5 años → evento moderado  
- T = 10 años → evento más extremo  

A mayor periodo de retorno, mayor es la magnitud de precipitación esperada.

## Interpolación lineal usando la tabla de resultados

Para estimar la precipitación asociada a un periodo de retorno específico, se utilizan los valores más cercanos de la tabla y se aplica interpolación lineal:

$$
P = P_1 + \frac{(T - T_1)}{(T_2 - T_1)} (P_2 - P_1)
$$


## Caso 1: T = 2 años

De la tabla:

- $T_1 = 2.000$, $P_1 = 55.2$ mm  

Como $T = 2$ coincide exactamente:

$P = 55.2$ mm  


## Caso 2: T = 5 años

De la tabla:

- $T_1 = 4.727$, $P_1 = 71.2$ mm  
- $T_2 = 5.200$, $P_2 = 72.7$ mm  

Aplicando la fórmula:

$$
P = 71.2 + \frac{(5 - 4.727)}{(5.200 - 4.727)} (72.7 - 71.2)
$$

$$
P = 71.2 + \frac{0.273}{0.473} \cdot 1.5
$$

$$
P \approx 71.2 + 0.87 = 72.1 \, mm
$$

$P \approx 72.1$ mm  



## Caso 3: T = 10 años

De la tabla:

- $T_1 = 8.667$, $P_1 = 81.8$ mm  
- $T_2 = 10.400$, $P_2 = 83.4$ mm  

$$
P = 81.8 + \frac{(10 - 8.667)}{(10.400 - 8.667)} (83.4 - 81.8)
$$

$$
P = 81.8 + \frac{1.333}{1.733} \cdot 1.6
$$

$$
P \approx 81.8 + 1.23 = 83.0 \, mm
$$

$P \approx 83.0$ mm  



- T = 2 años → evento frecuente (~55 mm)  
- T = 5 años → evento moderado (~72 mm)  
- T = 10 años → evento más extremo (~83 mm)  



Interpolación mediante código

In [ ]:

# ordenar de menor a mayor en T
df_interp = df_weibull.sort_values("T")

# periodos de retorno que queremos
T_objetivo = np.array([2, 5, 10])

# interpolación lineal en T
precipitaciones = np.interp(
    T_objetivo,
    df_interp["T"].values,
    df_interp["precipitacion_mm"].values
)

for T, P in zip(T_objetivo, precipitaciones):
    print(f"Para T = {T} años → Precipitación ≈ {P:.1f} mm")

## Cálculo de probabilidades a partir de la serie

Se estiman probabilidades empíricas utilizando la frecuencia relativa:

$$
P(A) = \frac{\text{número de casos que cumplen } A}{n}
$$

donde $n = 51$ es el número total de años.



### Probabilidad de precipitaciones menores a 50 mm

Número de casos: 16

$$
P(X < 50) = \frac{16}{51} = 0.3137
$$



### Probabilidad de precipitaciones mayores a 100 mm

Número de casos: 1

$$
P(X > 100) = \frac{1}{51} = 0.0196
$$



### Probabilidad de precipitaciones entre 50 y 500 mm

Número de casos: 35

$$
P(50 \leq X \leq 500) = \frac{35}{51} = 0.6863
$$



## Probabilidad de eventos consecutivos

Se asume independencia entre años.

Sea:

$$
p = P(X < 50) = 0.3137
$$



### Dos años consecutivos con precipitación menor a 50 mm

$$
P = p*p = p^2 = (0.3137)^2 = 0.0984
$$



### Diez años consecutivos con precipitación menor a 50 mm

$$
P = p^{10} = (0.3137)^{10} \approx 9.2 \times 10^{-6}
$$




- Los eventos con precipitación menor a 50 mm tienen una probabilidad moderada  
- Eventos extremos mayores a 100 mm son poco frecuentes  
- La probabilidad de múltiples años consecutivos con bajas precipitaciones disminuye rápidamente  

Esto refleja el comportamiento probabilístico de eventos independientes en el tiempo.

## Uso del complemento de probabilidad

También es posible calcular probabilidades usando el complemento:

$$
P(A^c) = 1 - P(A)
$$



### Probabilidad de precipitaciones mayores o iguales a 50 mm

Sabemos que:

$$
P(X < 50) = 0.3137
$$

Entonces:

$$
P(X \geq 50) = 1 - P(X < 50)
$$

$$
P(X \geq 50) = 1 - 0.3137 = 0.6863
$$





## Probabilidad de que NO ocurra un evento en n años consecutivos

Sea:

- $p = P(A)$ → probabilidad de que ocurra el evento en un año  
- $1 - p = P(A^c)$ → probabilidad de que NO ocurra el evento en un año  



### Supuesto

Se asume que los años son **independientes**.



### Probabilidad de no ocurrencia en n años

La probabilidad de que el evento **no ocurra durante n años consecutivos** es:

$$
P(\text{no ocurre en } n \text{ años}) = (1 - p)^n
$$



### Ejemplo

Si:

$$
p = P(X < 50) = 0.3137
$$

Entonces:

$$
P(\text{no ocurre } X<50 \text{ en 10 años}) = (1 - 0.3137)^{10}
$$

$$
= (0.6863)^{10} \approx 0.017
$$



- Existe una probabilidad de aproximadamente 1.7% de que durante 10 años consecutivos no se presenten precipitaciones menores a 50 mm  
